# Clasificación de sentimiento con BERT
### Enfoque 3 de 3 — Fine-tuning completo de BERT

**Universidad de Santiago de Chile (USACH)**  
Diplomado de Inteligencia artificial aplicada a la industria tecnológica  
Autor: Diego Fischer

Se hace **fine-tuning completo de BERT** (`trainable=True`): sus pesos se ajustan y sobre el vector `[CLS]` (`pooled_output`) se añade una capa densa. Con KerasHub usamos `keras_hub.models.BertBackbone` entrenable (o, en una línea, `keras_hub.models.BertTextClassifier`).

> Los tres cuadernos de esta serie resuelven la misma tarea (clasificación binaria de sentimiento sobre el dataset Sentiment140) y se diferencian únicamente en el rol que cumple BERT dentro del modelo.

> **Migrado a KerasHub.** El paquete `bert-for-tf2` quedó obsoleto y no funciona con Keras 3 (el *default* en Colab actual). Aquí usamos [KerasHub](https://keras.io/keras_hub/), la biblioteca nativa de Keras 3 para modelos preentrenados.

---

# Fase 1: Importar las dependencias

Instalamos **KerasHub**. Requiere Keras 3, que ya viene por defecto en Colab, así que no hace falta reiniciar el entorno.

In [ ]:
!pip install -q -U keras-hub

Importamos las librerías. Fijamos TensorFlow como *backend* de Keras 3 **antes** de importar `keras`. `keras_hub` nos da el tokenizador y el modelo BERT preentrenados.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"   # backend de Keras 3

import numpy as np
import math
import re
import pandas as pd
from bs4 import BeautifulSoup
import random

import tensorflow as tf
import keras
import keras_hub
from tensorflow.keras import layers

from google.colab import drive

# Preset de BERT preentrenado (equivalente a bert_en_uncased_L-12_H-768_A-12)
BERT_PRESET = "bert_base_en_uncased"

Montamos Google Drive para leer el dataset.

In [ ]:
drive.mount("/content/drive")

# Fase 2: Preprocesamiento de datos

## Carga de los ficheros

Leemos el CSV de entrenamiento de Sentiment140 (seis columnas; `encoding='latin1'` por los caracteres no UTF-8 de los tweets).

In [ ]:
cols = ["sentiment", "id", "date", "query", "user", "text"]
data = pd.read_csv(
    "/content/drive/My Drive/Curso de NLP/BERT/sentiment_data/training.csv",
    header=None,
    names=cols,
    engine="python",
    encoding="latin1"
)

Conservamos solo las columnas útiles: `sentiment` y `text`.

In [ ]:
data.drop(["id", "date", "query", "user"], axis=1, inplace=True)

## Limpieza

Función que limpia cada tweet: quita el HTML, las menciones `@usuario`, las URLs y todo lo que no sean letras o signos básicos, y colapsa los espacios sobrantes.

In [ ]:
def clean_tweet(tweet):
    tweet = BeautifulSoup(tweet, "lxml").get_text()
    # Eliminar el @ y su mención
    tweet = re.sub(r"@[A-Za-z0-9]+", ' ', tweet)
    # Eliminar los links de la URL
    tweet = re.sub(r"https?://[A-Za-z0-9./]+", ' ', tweet)
    # Conservar solo las letras y algunos signos
    tweet = re.sub(r"[^a-zA-Z.!?']", ' ', tweet)
    # Eliminar espacios en blanco adicionales
    tweet = re.sub(r" +", ' ', tweet)
    return tweet

Aplicamos la limpieza a todos los tweets.

In [ ]:
data_clean = [clean_tweet(tweet) for tweet in data.text]

Extraemos las etiquetas. En Sentiment140 el positivo viene como `4`; lo pasamos a `1` para tener etiquetas binarias **0 (negativo)** / **1 (positivo)**.

In [ ]:
data_labels = data.sentiment.values
data_labels[data_labels == 4] = 1

## Tokenización y empaquetado

Igual que en el enfoque anterior, usamos el preprocesador de BERT para obtener `{token_ids, segment_ids, padding_mask}` a longitud fija.

In [ ]:
SEQ_LEN = 40
preprocessor = keras_hub.models.BertTextClassifierPreprocessor.from_preset(
    BERT_PRESET, sequence_length=SEQ_LEN)

## Creación del dataset

Armamos el `tf.data.Dataset` y separamos entrenamiento/prueba.

In [ ]:
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

full_ds = tf.data.Dataset.from_tensor_slices((data_clean, data_labels))
full_ds = full_ds.shuffle(len(data_clean), seed=42)

NB_TEST = len(data_clean) // 10

def make_ds(ds):
    return (ds.batch(BATCH_SIZE)
              .map(lambda x, y: (preprocessor(x), y), num_parallel_calls=AUTOTUNE)
              .prefetch(AUTOTUNE))

test_dataset  = make_ds(full_ds.take(NB_TEST))
train_dataset = make_ds(full_ds.skip(NB_TEST))

> El *fine-tuning* de BERT es **caro**: ajusta los ~110 M de parámetros del modelo. Sobre 1,6 M de tweets tardaría muchísimo, así que entrenamos sobre un subconjunto reducido (como en el cuaderno original).

In [ ]:
train_dataset_light = train_dataset.take(500)

# Fase 3: Construcción del modelo

Tomamos el `pooled_output` de BERT (el vector `[CLS]`, un resumen de toda la oración), le aplicamos *dropout* y una capa densa que entrega los *logits* de las 2 clases. Aquí **BERT es entrenable** (`trainable=True` por defecto), así que sus pesos se ajustan al problema.

> Alternativa en una línea: KerasHub trae un clasificador BERT completo, `keras_hub.models.BertTextClassifier.from_preset(BERT_PRESET, num_classes=2)`, que hace exactamente esto (backbone + cabeza). Lo escribimos explícito para ver las partes.

In [ ]:
class BERTClassifier(tf.keras.Model):
    def __init__(self, bert_preset, nb_classes=2, dropout_rate=0.1,
                 name="bert_classifier"):
        super().__init__(name=name)
        self.bert = keras_hub.models.BertBackbone.from_preset(bert_preset)  # entrenable
        self.dropout = layers.Dropout(dropout_rate)
        self.last_dense = layers.Dense(nb_classes)   # logits (sin softmax)

    def call(self, inputs, training=False):
        pooled = self.bert(inputs)["pooled_output"]   # (batch, 768) -> vector [CLS]
        pooled = self.dropout(pooled, training=training)
        return self.last_dense(pooled)

# Fase 4: Entrenamiento

Compilamos con Adam y una tasa de aprendizaje baja (5e-5), la recomendada para *fine-tuning* de BERT. Como la última capa entrega *logits*, usamos `from_logits=True`.

In [ ]:
NB_CLASSES = 2
DROPOUT_RATE = 0.1
NB_EPOCHS = 5
INIT_LR = 5e-5

bert_classifier = BERTClassifier(BERT_PRESET, nb_classes=NB_CLASSES,
                                 dropout_rate=DROPOUT_RATE)

bert_classifier.compile(
    optimizer=keras.optimizers.Adam(INIT_LR),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy()])

Guardamos los pesos en Drive por época.

In [ ]:
checkpoint_path = "/content/drive/My Drive/Curso de NLP/BERT/ckpt_bert_class.weights.h5"
ckpt_cb = keras.callbacks.ModelCheckpoint(checkpoint_path, save_weights_only=True)

Entrenamos sobre el subconjunto reducido.

In [ ]:
bert_classifier.fit(train_dataset_light, epochs=NB_EPOCHS, callbacks=[ckpt_cb])

# Fase 5: Evaluación

In [ ]:
bert_classifier.evaluate(test_dataset)

Probamos con una frase nueva. La salida son *logits*; la clase es el `argmax`.

In [ ]:
def get_prediction(sentence):
    inputs = preprocessor([sentence])
    logits = bert_classifier(inputs, training=False)
    sentiment = int(tf.argmax(tf.squeeze(logits)).numpy())
    etiqueta = "Positivo" if sentiment == 1 else "Negativo"
    print("Logits: {}\nSentimiento predicho: {}.".format(logits.numpy(), etiqueta))

get_prediction("I love being with you")

---
## Licencia

[![Licencia Creative Commons](https://i.creativecommons.org/l/by/4.0/88x31.png)](https://creativecommons.org/licenses/by/4.0/deed.es)

Esta obra está licenciada bajo una [Licencia Creative Commons Atribución 4.0 Internacional (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/deed.es).

**Atribución:** Diego Fischer — Universidad de Santiago de Chile (USACH).